In [ ]:
# ==================================================
# Imports
# ==================================================
from datetime import datetime as td
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3

In [ ]:
# ==================================================
# Project Configuration
# ==================================================
URL = "https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks"
CSV_PATH = "exchange_rate.csv"
OUTPUT_PATH = "./Largest_banks_data.csv"
DATABASE_NAME = "Banks.db"
TABLE_NAME = "Largest_banks"
LOG_FILE = "code_log.txt"

In [ ]:
def log_progress(message):
  """
  Log the progress of the ETL pipeline with a timestamp.
  """
  timestamp_format = "%Y-%m-%d %H:%M:%S"
  now = td.now()
  timestamp = now.strftime(timestamp_format)
  with open(LOG_FILE,"a") as log_file:
    log_file.write(f"{timestamp}, {message}\n")

In [ ]:
def extract(url,table_attribs = None ):
  """
  Extract the largest banks data from the archived Wikipedia page
  and return it as a pandas DataFrame.
  """
  page_response = requests.get(url)

  #Check the status of the response
  if page_response.status_code != 200:
    raise Exception("Failed to retrieve the web page.")

  # Parse the HTML page
  soup = BeautifulSoup(page_response.text,"html.parser")

  # Find the Market Capitalization table
  target_table = soup.find(
      "table",
      class_ ="wikitable sortable mw-collapsible"
  )

  # Create empty lists
  bank_names  = []
  market_caps = []

  rows = target_table.find_all("tr")
  for row in rows[1:]:
    columns = row.find_all("td")

    bank_names.append(columns[1].get_text(strip=True))
    market_caps.append(columns[2].get_text(strip=True))


  df = pd.DataFrame({
      "Name": bank_names,
      "MC_USD_Billion":market_caps
  })

  return df

extract(URL)


,Name,MC_USD_Billion
0,JPMorgan Chase,432.92
1,Bank of America,231.52
2,Industrial and Commercial Bank of China,194.56
3,Agricultural Bank of China,160.68
4,HDFC Bank,157.91
5,Wells Fargo,155.87
6,HSBC Holdings PLC,148.90
7,Morgan Stanley,140.83
8,China Construction Bank,139.82
9,Bank of China,136.81


In [ ]:
def transform(df,csv_path):
  """
  Convert market capitalization from USD to GBP, EUR, and INR
  using the provided exchange rates.
  """
  df['MC_USD_Billion'] = (
      df['MC_USD_Billion']
      .str.replace("$", "", regex=False)
      .str.replace("B", "", regex=False)
      .str.replace(",", "", regex=False)
      .astype(float)
  )


  exchange_rates = pd.read_csv(csv_path,index_col=0)

  gbp_rate = exchange_rates.loc['GBP',"Rate"]
  eur_rate = exchange_rates.loc['EUR',"Rate"]
  inr_rate = exchange_rates.loc['INR',"Rate"]

  df['MC_GBP_Billion'] = round(df['MC_USD_Billion'] * gbp_rate , 2)
  df['MC_EUR_Billion'] = round(df['MC_USD_Billion'] * eur_rate , 2)
  df['MC_INR_Billion'] = round(df['MC_USD_Billion'] * inr_rate ,2)

  return df

In [ ]:
def load_to_csv(df,output_path):
  """
  Save the transformed DataFrame to a CSV file.
  """
  df.to_csv(output_path,index=False)
  log_progress(f"Data saved to CSV")

In [ ]:
def load_to_db(df,conn,table_name):
  """
  Load the DataFrame into an SQLite database table.
  """
  df.to_sql(table_name,conn,if_exists="replace",index= False)
  log_progress("DataFrame saved successfully to database")


In [ ]:
def run_query(query_statement, sql_connection):
    """
    Execute a SQL query on the database, print the result,
    and log the executed query.
    """

    result = pd.read_sql_query(query_statement, sql_connection)
    print(result)

    log_progress(f"Executed query: {query_statement}")
    return result

## **Main Program**

In [ ]:
log_progress("ETL Job Started")

log_progress("Extract phase Started")
df = extract(URL)
log_progress("Extract phase Ended")

log_progress("Transform phase Started")
df = transform(df,CSV_PATH)
log_progress("Transform phase Ended")


log_progress("Load phase Started")
load_to_csv(df,OUTPUT_PATH)
log_progress("Load phase Ended")


log_progress("DB Connect phase Started")
conn = sqlite3.connect(DATABASE_NAME)
log_progress("DB Connect phase Ended")

log_progress("DB Load phase Started")
load_to_db(df, conn, TABLE_NAME)
log_progress("DB Load phase Ended")

log_progress("DB Query phase Started")
print(run_query("SELECT * FROM Largest_banks WHERE Name = 'JPMorgan Chase'", conn))
log_progress("DB Query phase Ended")

conn.close()

log_progress("ETL Job Ended")


             Name  MC_USD_Billion  MC_GBP_Billion  MC_EUR_Billion  \
0  JPMorgan Chase          432.92          402.62          402.62   

   MC_INR_Billion  
0        35910.71  
